In [ ]:
# ============================================================
# YOLO11n VEHICLE TRACKING + APPROXIMATE SPEED
# USING road_calibration.json
# ============================================================

import cv2
import json
import os
import numpy as np

from collections import defaultdict, deque
from ultralytics import YOLO
from IPython.display import Video, display


# ============================================================
# 1. PATHS
# ============================================================

VIDEO_PATH = r"C:\Users\PC-LAB1\Desktop\New folder\Test Video.mp4"

MODEL_PATH = (
    r"C:\Users\PC-LAB1\Desktop\shok"
    r"\notebooks\Day_09\speed\yolo11n.pt"
)

CALIBRATION_FILE = (
    r"C:\Users\PC-LAB1\Desktop\New folder"
    r"\calibration\road_calibration.json"
)

OUTPUT_VIDEO = (
    r"C:\Users\PC-LAB1\Desktop\New folder"
    r"\Test Video_speed_tracking.mp4"
)


# ============================================================
# 2. APPROXIMATE ROAD MEASUREMENTS
# ============================================================
#
# IMPORTANT:
# These are physical estimates, not automatically obtained
# from the image.
#
# Change these if you know the actual road dimensions.
# ============================================================

ROAD_WIDTH_M = 10.5

ROAD_LENGTH_M = 80.0


# ============================================================
# 3. SPEED SETTINGS
# ============================================================

MIN_HISTORY = 5

SPEED_WINDOW = 8

MAX_SPEED_KMH = 150.0

SPEED_SMOOTHING = 0.25


# ============================================================
# 4. VEHICLE CLASSES
# ============================================================

VEHICLE_CLASSES = {
    2: "car",
    3: "motorcycle",
    5: "bus",
    7: "truck"
}


# ============================================================
# 5. CHECK FILES
# ============================================================

for path, name in [
    (VIDEO_PATH, "Video"),
    (MODEL_PATH, "YOLO model"),
    (CALIBRATION_FILE, "Calibration")
]:

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"{name} not found:\n{path}"
        )


# ============================================================
# 6. LOAD YOLO11n
# ============================================================

print("Loading YOLO11n...")

model = YOLO(
    MODEL_PATH
)

print("YOLO11n loaded.")


# ============================================================
# 7. LOAD ROAD CALIBRATION
# ============================================================

with open(
    CALIBRATION_FILE,
    "r"
) as f:

    calibration = json.load(f)


left_boundary = np.array(
    calibration["left_boundary"],
    dtype=np.float32
)

right_boundary = np.array(
    calibration["right_boundary"],
    dtype=np.float32
)


print()
print("Calibration loaded.")

print(
    "Left boundary points:",
    len(left_boundary)
)

print(
    "Right boundary points:",
    len(right_boundary)
)


# ============================================================
# 8. LOAD VIDEO
# ============================================================

cap = cv2.VideoCapture(
    VIDEO_PATH
)

if not cap.isOpened():

    raise RuntimeError(
        "Could not open video."
    )


FPS = cap.get(
    cv2.CAP_PROP_FPS
)

FRAME_COUNT = int(
    cap.get(
        cv2.CAP_PROP_FRAME_COUNT
    )
)

WIDTH = int(
    cap.get(
        cv2.CAP_PROP_FRAME_WIDTH
    )
)

HEIGHT = int(
    cap.get(
        cv2.CAP_PROP_FRAME_HEIGHT
    )
)


print()
print("========================================")
print("VIDEO")
print("========================================")

print(
    f"Resolution : {WIDTH} x {HEIGHT}"
)

print(
    f"FPS        : {FPS:.2f}"
)

print(
    f"Frames     : {FRAME_COUNT}"
)

print(
    f"Duration   : {FRAME_COUNT / FPS:.2f} sec"
)


# ============================================================
# 9. PREPARE ROAD BOUNDARIES
# ============================================================

def prepare_boundary(points):

    points = np.asarray(
        points,
        dtype=np.float32
    )

    # Sort by Y coordinate
    order = np.argsort(
        points[:, 1]
    )

    points = points[order]

    y = points[:, 1]
    x = points[:, 0]

    # Remove duplicate Y values
    unique_y, indices = np.unique(
        y,
        return_index=True
    )

    unique_x = x[indices]

    return unique_y, unique_x


left_y, left_x = prepare_boundary(
    left_boundary
)

right_y, right_x = prepare_boundary(
    right_boundary
)


road_y_min = max(
    left_y.min(),
    right_y.min()
)

road_y_max = min(
    left_y.max(),
    right_y.max()
)


# ============================================================
# 10. ROAD EDGE INTERPOLATION
# ============================================================

def road_edges_at_y(y):

    y = float(y)

    y = np.clip(
        y,
        road_y_min,
        road_y_max
    )

    x_left = np.interp(
        y,
        left_y,
        left_x
    )

    x_right = np.interp(
        y,
        right_y,
        right_x
    )

    return x_left, x_right


# ============================================================
# 11. CONVERT IMAGE POSITION TO APPROXIMATE ROAD POSITION
# ============================================================

def get_road_position(x, y):

    x_left, x_right = road_edges_at_y(
        y
    )

    width_pixels = (
        x_right - x_left
    )

    if width_pixels <= 2:

        return None


    # Position across road
    lateral = (
        x - x_left
    ) / width_pixels


    # Position along road
    normalized_y = (
        road_y_max - y
    ) / (
        road_y_max - road_y_min
    )


    normalized_y = np.clip(
        normalized_y,
        0.0,
        1.0
    )


    # Perspective correction
    longitudinal = (
        normalized_y ** 1.65
    ) * ROAD_LENGTH_M


    return (
        lateral,
        longitudinal
    )


# ============================================================
# 12. TRACK HISTORY
# ============================================================

track_history = defaultdict(
    lambda: deque(
        maxlen=30
    )
)

speed_history = {}

latest_speed = {}

all_track_ids = set()

speed_estimate_count = defaultdict(
    int
)

all_speed_values = []


# ============================================================
# 13. CREATE OUTPUT VIDEO
# ============================================================

os.makedirs(
    os.path.dirname(
        OUTPUT_VIDEO
    ),
    exist_ok=True
)


# H.264 may not be available.
# mp4v is widely supported by OpenCV.

fourcc = cv2.VideoWriter_fourcc(
    *"mp4v"
)


writer = cv2.VideoWriter(
    OUTPUT_VIDEO,
    fourcc,
    FPS,
    (WIDTH, HEIGHT)
)


if not writer.isOpened():

    raise RuntimeError(
        "Could not create output video."
    )


# ============================================================
# 14. DRAW ROAD BOUNDARIES
# ============================================================

def draw_boundaries(frame):

    # LEFT = RED

    left_pts = (
        left_boundary
        .astype(np.int32)
        .reshape((-1, 1, 2))
    )

    cv2.polylines(
        frame,
        [left_pts],
        False,
        (0, 0, 255),
        3
    )


    # RIGHT = BLUE

    right_pts = (
        right_boundary
        .astype(np.int32)
        .reshape((-1, 1, 2))
    )

    cv2.polylines(
        frame,
        [right_pts],
        False,
        (255, 0, 0),
        3
    )


# ============================================================
# 15. SPEED CALCULATION
# ============================================================

def calculate_speed(history):

    if len(history) < MIN_HISTORY:

        return None


    if len(history) < SPEED_WINDOW:

        return None


    first = history[
        -SPEED_WINDOW
    ]

    last = history[-1]


    frame1 = first[0]

    frame2 = last[0]


    position1 = first[3]

    position2 = last[3]


    if (
        position1 is None
        or
        position2 is None
    ):

        return None


    frame_difference = (
        frame2 - frame1
    )


    if frame_difference <= 0:

        return None


    time_seconds = (
        frame_difference / FPS
    )


    distance_m = abs(
        position2[1]
        -
        position1[1]
    )


    speed_mps = (
        distance_m /
        time_seconds
    )


    speed_kmh = (
        speed_mps * 3.6
    )


    if speed_kmh > MAX_SPEED_KMH:

        return None


    return speed_kmh


# ============================================================
# 16. PROCESS VIDEO
# ============================================================

frame_number = 0


print()
print("========================================")
print("STARTING TRACKING")
print("========================================")


while True:

    ret, frame = cap.read()


    if not ret:

        break


    frame_number += 1


    # --------------------------------------------------------
    # YOLO11n TRACKING
    # --------------------------------------------------------

    results = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        classes=list(
            VEHICLE_CLASSES.keys()
        ),
        conf=0.20,
        iou=0.50,
        verbose=False
    )


    output = frame.copy()


    # --------------------------------------------------------
    # DRAW ROAD
    # --------------------------------------------------------

    draw_boundaries(
        output
    )


    # --------------------------------------------------------
    # VEHICLE DETECTIONS
    # --------------------------------------------------------

    if (
        len(results) > 0
        and
        results[0].boxes is not None
        and
        results[0].boxes.id is not None
    ):

        boxes = results[0].boxes


        xyxy = (
            boxes.xyxy
            .cpu()
            .numpy()
        )


        ids = (
            boxes.id
            .cpu()
            .numpy()
            .astype(int)
        )


        classes = (
            boxes.cls
            .cpu()
            .numpy()
            .astype(int)
        )


        confidences = (
            boxes.conf
            .cpu()
            .numpy()
        )


        for (
            box,
            track_id,
            cls,
            confidence
        ) in zip(
            xyxy,
            ids,
            classes,
            confidences
        ):


            if cls not in VEHICLE_CLASSES:

                continue


            all_track_ids.add(
                track_id
            )


            x1, y1, x2, y2 = map(
                int,
                box
            )


            # ------------------------------------------------
            # Bottom-center point
            # ------------------------------------------------

            center_x = (
                x1 + x2
            ) // 2


            bottom_y = y2


            # ------------------------------------------------
            # Road position
            # ------------------------------------------------

            road_position = get_road_position(
                center_x,
                bottom_y
            )


            # ------------------------------------------------
            # Save history
            # ------------------------------------------------

            track_history[
                track_id
            ].append(
                (
                    frame_number,
                    center_x,
                    bottom_y,
                    road_position
                )
            )


            # ------------------------------------------------
            # Calculate speed
            # ------------------------------------------------

            speed = calculate_speed(
                track_history[track_id]
            )


            if speed is not None:

                if track_id in speed_history:

                    speed = (
                        SPEED_SMOOTHING * speed
                        +
                        (1 - SPEED_SMOOTHING)
                        * speed_history[track_id]
                    )


                speed_history[
                    track_id
                ] = speed


                latest_speed[
                    track_id
                ] = speed


                speed_estimate_count[
                    track_id
                ] += 1


                all_speed_values.append(
                    speed
                )


            # ------------------------------------------------
            # Bounding box
            # ------------------------------------------------

            cv2.rectangle(
                output,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                2
            )


            # ------------------------------------------------
            # Bottom-center point
            # ------------------------------------------------

            cv2.circle(
                output,
                (
                    center_x,
                    bottom_y
                ),
                5,
                (0, 255, 255),
                -1
            )


            # ------------------------------------------------
            # Label
            # ------------------------------------------------

            vehicle_name = (
                VEHICLE_CLASSES[cls]
            )


            label = (
                f"ID {track_id} "
                f"{vehicle_name}"
            )


            if track_id in latest_speed:

                label += (
                    f" "
                    f"{latest_speed[track_id]:.1f}"
                    f" km/h"
                )


            # ------------------------------------------------
            # Label background
            # ------------------------------------------------

            (
                text_width,
                text_height
            ), baseline = cv2.getTextSize(
                label,
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                2
            )


            text_x = x1

            text_y = max(
                y1 - 8,
                text_height + 5
            )


            cv2.rectangle(
                output,
                (
                    text_x,
                    text_y - text_height - baseline
                ),
                (
                    text_x + text_width,
                    text_y + baseline
                ),
                (0, 0, 0),
                -1
            )


            cv2.putText(
                output,
                label,
                (
                    text_x,
                    text_y
                ),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (255, 255, 255),
                2,
                cv2.LINE_AA
            )


    # ========================================================
    # INFORMATION
    # ========================================================

    info = (
        f"Frame: "
        f"{frame_number}/{FRAME_COUNT}"
    )


    info2 = (
        f"Tracked vehicles: "
        f"{len(all_track_ids)}"
    )


    info3 = (
        f"Speed tracks: "
        f"{len(speed_estimate_count)}"
    )


    cv2.rectangle(
        output,
        (5, 5),
        (330, 82),
        (0, 0, 0),
        -1
    )


    cv2.putText(
        output,
        info,
        (12, 27),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (255, 255, 255),
        1,
        cv2.LINE_AA
    )


    cv2.putText(
        output,
        info2,
        (12, 50),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (255, 255, 255),
        1,
        cv2.LINE_AA
    )


    cv2.putText(
        output,
        info3,
        (12, 73),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (255, 255, 255),
        1,
        cv2.LINE_AA
    )


    # ========================================================
    # WRITE FRAME
    # ========================================================

    writer.write(
        output
    )


    # ========================================================
    # PROGRESS
    # ========================================================

    if (
        frame_number % 10 == 0
        or
        frame_number == FRAME_COUNT
    ):

        print(
            f"Processed "
            f"{frame_number}/{FRAME_COUNT}",
            end="\r"
        )


# ============================================================
# CLEANUP
# ============================================================

cap.release()

writer.release()


# ============================================================
# RESULTS
# ============================================================

print()
print()

print("========================================")
print("TRACKING COMPLETE")
print("========================================")

print(
    f"Total tracked vehicles: "
    f"{len(all_track_ids)}"
)

print(
    f"Vehicles with speed: "
    f"{len(speed_estimate_count)}"
)

print()
print(
    "Output:"
)

print(
    OUTPUT_VIDEO
)


# ============================================================
# SPEED SUMMARY
# ============================================================

if len(all_speed_values) > 0:

    speeds = np.array(
        all_speed_values
    )


    print()
    print("========================================")
    print("SPEED SUMMARY")
    print("========================================")


    print(
        f"Speed measurements: "
        f"{len(speeds)}"
    )


    print(
        f"Minimum: "
        f"{speeds.min():.1f} km/h"
    )


    print(
        f"Maximum: "
        f"{speeds.max():.1f} km/h"
    )


    print(
        f"Average: "
        f"{speeds.mean():.1f} km/h"
    )


    print(
        f"Median: "
        f"{np.median(speeds):.1f} km/h"
    )


# ============================================================
# DISPLAY VIDEO
# ============================================================

print()
print("Displaying output video...")

display(
    Video(
        OUTPUT_VIDEO,
        embed=True,
        width=960
    )
)